This notebook explores, extracts, and loads data from the Photovoltaic Data Acquisition (PVDAQ) Public Datasets, part of the US Department of Energy Open Energy Data Initiative.

The PVDAQ Public Data Lake solar PV data are hosted on Amazon S3 at 's3://oedi-data-lake/pvdaq/parquet/'. 

Within this /parquet/ directory are eight subdirectories that correspond to metadata lookup tables: inverters, meters, metrics, mount, other_instruments, site, system

Also within /parquet/ is a subdirectory, pvdata, that contains the time series data itself, partitioned by system_id, year, month, and day.

In [0]:
%sql

-- =============================================================================
-- Loading metadata / lookup tables from the S3 parquet files
-- =============================================================================

CREATE CATALOG IF NOT EXISTS pvdaq_catalog;
CREATE SCHEMA IF NOT EXISTS pvdaq_catalog.bronze;

USE CATALOG pvdaq_catalog;
USE SCHEMA bronze;
SELECT current_catalog(), current_schema();

-- First loading the raw tables with SELECT *, and then manually reordering the table schemas according to the PVDAQ documentation

CREATE OR REPLACE TABLE inverters AS
WITH raw_inverters AS (
  SELECT *
  FROM read_files(
    's3://oedi-data-lake/pvdaq/parquet/inverters/',
    format => 'parquet'
  )
)
SELECT 
  inverter_id,
  name,
  manufacturer,
  model,
  serial_num,
  num_strings,
  modules_per_string,
  type,
  quantity,
  time_interval,
  site_id,
  system_id,
  comments
FROM raw_inverters;

CREATE OR REPLACE TABLE meters AS
WITH raw_meters AS (
  SELECT *
  FROM read_files(
    's3://oedi-data-lake/pvdaq/parquet/meters/',
    format => 'parquet'
  )
)
SELECT 
  meter_id,
  name,
  manufacturer,
  model,
  serial_num,
  time_interval,
  type,
  site_id,
  system_id,
  comments
FROM raw_meters;

CREATE OR REPLACE TABLE metrics AS
WITH raw_metrics AS (
  SELECT *
  FROM read_files(
    's3://oedi-data-lake/pvdaq/parquet/metrics/',
    format => 'parquet'
  )
)
SELECT 
  system_id,
  metric_id,
  sensor_name,
  common_name,
  raw_units,
  units,
  calc_scale,
  calc_offset,
  calc_details,
  aggregation_type,
  source_type,
  source_id,
  comments,
  standard_name
FROM raw_metrics;

CREATE OR REPLACE TABLE modules AS
WITH raw_modules AS (
  SELECT *
  FROM read_files(
    's3://oedi-data-lake/pvdaq/parquet/modules/',
    format => 'parquet'
  )
)
SELECT 
  module_id,
  name,
  inverter_id,
  manufacturer,
  model,
  serial_num,
  type,
  quantity,
  reference_module,
  start_on,
  end_on,
  site_id,
  system_id,
  comments
FROM raw_modules;

CREATE OR REPLACE TABLE mount AS
WITH raw_mount AS (
  SELECT *
  FROM read_files(
    's3://oedi-data-lake/pvdaq/parquet/mount/',
    format => 'parquet'
  )
)
SELECT 
  mount_id,
  name,
  manufacturer,
  model,
  azimuth,
  tilt,
  tracking,
  type,
  site_id,
  system_id
  -- schema documentation drift: comments not present in mount table
FROM raw_mount;

CREATE OR REPLACE TABLE other_instruments AS
WITH raw_other_instruments AS (
  SELECT *
  FROM read_files(
    's3://oedi-data-lake/pvdaq/parquet/other-instruments/',
    format => 'parquet'
  )
)
SELECT 
  instrument_id,
  name,
  manufacturer,
  model,
  serial_num,
  time_interval,
  type,
  site_id,
  system_id,
  comments
FROM raw_other_instruments;

CREATE OR REPLACE TABLE site AS
WITH raw_site AS (
  SELECT *
  FROM read_files(
    's3://oedi-data-lake/pvdaq/parquet/site/',
    format => 'parquet'
  )
)
SELECT 
  site_id,
  system_id,
  public_name,
  location,
  latitude,
  longitude,
  elevation,
  av_pressure,
  av_temp,
  climate_type
FROM raw_site;

CREATE OR REPLACE TABLE system AS
WITH raw_system AS (
  SELECT *
  FROM read_files(
    's3://oedi-data-lake/pvdaq/parquet/system/',
    format => 'parquet'
  )
)
SELECT 
  system_id,
  site_id,
  public_name,
  area,
  power,
  started_on,
  ended_on,
  comments
FROM raw_system;


In [0]:
%sql

-- =============================================================================
-- Exploration and selection of the time series data that are eligible for 
-- analysis with the PVAnalytics Python library
-- =============================================================================

-- Estimating the size of the time series dataset (/pvdata/)

-- This is a deeply partitioned S3 dataset, partitioned into /pvdata/system_id/year/month/day, with rows typically at a 15 minute resolution per day

SELECT COUNT(*) AS single_month_rows
FROM read_files(
  's3a://oedi-data-lake/pvdaq/parquet/pvdata/system_id=10/year=2020/month=1/day=23',
  format => 'parquet'
);
-- 10,640 rows for a single day

SELECT COUNT(*) 
  FROM system;
-- 157 systems in total

-- With 157 systems, and several years of data for each system, there are likely multiple billions of rows in this dataset

-- The schema of the time series data table is: system_id, year, month, day, measured_on, utc_measured_on, metric_id, value

-- We need to be able to join the time series data with metadata tables on the metric_id in order to calculate certain metrics for PV systems over time

-- Selecting row counts of metadata tables to inspect
SELECT 'pvdaq_site' AS table_name, COUNT(*) AS row_count FROM site
UNION ALL
SELECT 'pvdaq_system' AS table_name, COUNT(*) AS row_count FROM system
UNION ALL
SELECT 'pvdaq_inverters' AS table_name, COUNT(*) AS row_count FROM inverters
UNION ALL
SELECT 'pvdaq_meters' AS table_name, COUNT(*) AS row_count FROM meters
UNION ALL
SELECT 'pvdaq_modules' AS table_name, COUNT(*) AS row_count FROM modules
UNION ALL
SELECT 'pvdaq_mount' AS table_name, COUNT(*) AS row_count FROM mount
UNION ALL
SELECT 'pvdaq_other_instruments' AS table_name, COUNT(*) AS row_count FROM other_instruments
UNION ALL
SELECT 'pvdaq_metrics' AS table_name, COUNT(*) AS row_count FROM metrics
ORDER BY table_name;

-- The above query shows that there are 1,768 unique values of metric_id. The PVDAQ documentation explains that sensor_name is "referenced name produced by the instrumentation or tagged by array owner." metric_id is a system-specific sensor key.

-- It seems that the researchers and developers of this data lake helpfully performed some grouping of the sensor_names under the field common_name, which is defined as "a general grouping of sensor types (e.g. DC voltage, AC energy, POA irradiance)." 

SELECT DISTINCT common_name FROM metrics;

-- As can be seen from the above query, there are 38 distinct common_name strings. These are the useful measurements that can be compared across PV systems, such as AC power, POA irradiance, wind speed, etc.

-- I want to use the NREL developed Python library PVAnalytics to perform data quality processes and evaluate PV system performance. For the purposes of this project the relevant functions require four metrics: AC power, Irradiance POA, Temperature ambient, and Wind Speed. So I will only extract data from systems that have data on all of these metrics, in order to present an analysis with quality data.

-- Find all system_ids in pvdaq_metrics that have all 4 required metrics
SELECT system_id
FROM metrics
WHERE common_name IN (
  'AC power',
  'Irradiance POA',
  'Temperature ambient',
  'Wind speed'
)
GROUP BY system_id
HAVING COUNT(DISTINCT common_name) = 4
ORDER BY system_id;
-- 23 systems have the 4 metrics

-- In order to narrow the size of the time series dataset I will extract, necessary for the tractability of this project hosted on the Databricks Free Edition, I will choose to focus on only 2020 data. 

-- Identify only systems that have the four required metrics AND data present in 2020
SELECT system_id
FROM metrics
WHERE common_name IN (
    'AC power',
    'Irradiance POA',
    'Temperature ambient',
    'Wind speed'
)
GROUP BY system_id
HAVING COUNT(DISTINCT common_name) = 4

INTERSECT

SELECT DISTINCT system_id
FROM read_files(
    's3a://oedi-data-lake/pvdaq/parquet/pvdata/system_id=*/year=2020/*/*/*.parquet',
    format => 'parquet'
)
ORDER BY CAST(system_id AS INT);

-- Selected systems: 34, 35, 1200, 1201, 1202, 1239, 1276, 1277, 1278, 1283, 1367, 1418, 1419

-- Calculating total number of rows for the selected systems
SELECT 
  system_id,
  COUNT(*) AS total_system_rows
FROM read_files(
  's3a://oedi-data-lake/pvdaq/parquet/pvdata/system_id={34,35,1200,1201,1202,1239,1276,1277,1278,1283,1367,1418,1419}/year=2020/*/*/*.parquet',
  format => 'parquet'
)
GROUP BY system_id
ORDER BY CAST(system_id AS INT);
-- Total rows: 44,042,804


In [0]:
%sql
-- =============================================================================
-- Ingesting the 2020 data from the 13 systems into Delta Lake
-- =============================================================================

USE CATALOG pvdaq_catalog;
USE SCHEMA bronze;

-- Creating the Bronze table
CREATE TABLE IF NOT EXISTS pvdata_2020_sample (
  system_id INT,
  year INT,
  month INT,
  day INT,
  measured_on TIMESTAMP,
  utc_measured_on TIMESTAMP,
  metric_id INT,
  value DOUBLE
);

-- Run idempotent COPY INTO with metadata path extraction
-- Use regular expressions to extract the system_id values out of the partitioned parquet files; This will allow later joining on the metadata tables
COPY INTO pvdata_2020_sample
FROM (
  SELECT 
    CAST(regexp_extract(_metadata.file_path, 'system_id=([0-9]+)', 1) AS INT) AS system_id,
    CAST(regexp_extract(_metadata.file_path, 'year=([0-9]+)', 1) AS INT) AS year,
    CAST(regexp_extract(_metadata.file_path, 'month=([0-9]+)', 1) AS INT) AS month,
    CAST(regexp_extract(_metadata.file_path, 'day=([0-9]+)', 1) AS INT) AS day,
    measured_on,
    utc_measured_on,
    metric_id,
    value
  FROM 's3a://oedi-data-lake/pvdaq/parquet/pvdata/system_id={34,35,1200,1201,1202,1239,1276,1277,1278,1283,1367,1418,1419}/year=2020/*/*/*.parquet' 
  -- just extract the selected systems
)
FILEFORMAT = PARQUET;

-- Check table size
SELECT COUNT(*) FROM pvdata_2020_sample;
-- 44,042,804 rows loaded

